In [1]:
import mediapipe as mp
import cv2
import pydirectinput

In [12]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands


In [16]:
import cv2
import mediapipe as mp
import pydirectinput

cap = cv2.VideoCapture(0)
cap.set(3, 560)
cap.set(4, 400)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

with mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.5, 
    min_tracking_confidence=0.5
) as hands:
    
    pose = 'idle'

    while cap.isOpened():
        success, img = cap.read()
        img = cv2.flip(img, 1)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        height, width, _ = img.shape
        x1 = width//3
        x2 = x1*2

        try:
            hand_landmarks = results.multi_hand_landmarks[0]
            wrist = hand_landmarks.landmark[
                mp_hands.HandLandmark.WRIST
            ]
            
            right_hand = (wrist.x * width, wrist.y * height)

            pose = 'move'
            if( right_hand[0] < x1 ):
                pose = 'LEFT'
                pydirectinput.keyDown('left')
                pydirectinput.keyUp('right')
            elif(right_hand[0] > x2):
                pose = 'RIGHT'
                pydirectinput.keyDown('right')
                pydirectinput.keyUp('left')
            else:
                pose = "CENTER"
                pydirectinput.keyUp('left')
                pydirectinput.keyUp('right')

        except:
            pose = "No Detection"

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    img, hand_landmarks, mp_hands.HAND_CONNECTIONS
                )

        cv2.putText(img, pose, (50, 50), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 0), 5)
        cv2.line(img, (x1, 0), (x1, height), (255, 0, 0), 2)
        cv2.line(img, (x2, 0), (x2, height), (255, 0, 0), 2)

        cv2.imshow("Car Game", img)
        cv2.setWindowProperty("Car Game", cv2.WND_PROP_TOPMOST, 1)
            
        if cv2.waitKey(1) == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()